In [ ]:
!pip install optuna -q
!pip install --upgrade lightgbm -q
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 82.6 MB/s eta 0:00:00


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from google.colab import drive

from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score

import lightgbm as lgb
import xgboost as xgb
import optuna

drive.mount('/content/drive', force_remount=True)

set_config(transform_output="pandas")

Mounted at /content/drive


In [ ]:
PROJECT_PATH = '/content/drive/MyDrive/Colab Notebooks/Home_Credit_Project'

X_train_full = pd.read_parquet(f'{PROJECT_PATH}/X_train_full.parquet')
y_train = pd.read_parquet(f'{PROJECT_PATH}/y_train.parquet')['TARGET']

In [ ]:
skf = StratifiedKFold(n_splits = 5, shuffle=True, random_state=42)

In [ ]:
pcb_cols = [ col for col in X_train_full if col.startswith("PREV_")]
bereua_cols = [ col for col in X_train_full if col.startswith("BUREAU_") and not col.startswith("BUREAU_BB_")]
custom_fill_na_cols = pcb_cols + bereua_cols

In [ ]:
def fillna_col (df):
    return df.fillna(0)
num_zero_transform = FunctionTransformer(fillna_col)
def to_category(df):
    return df.astype("category")
def fillna_category(df):
    return df.fillna("None")
cat_with_na = Pipeline([
    ("fill_na", FunctionTransformer(fillna_category)),
    ("to_category", FunctionTransformer(to_category))
])
def car_age(df):
    return df.fillna(-1)
fill_car_age = FunctionTransformer(car_age)
def fix_day_employed(df):
    df = df.copy()
    if "DAYS_EMPLOYED" in df.columns:
        df["DE_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
        df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace({365243: np.nan})
    return df
fix_de = FunctionTransformer(fix_day_employed)

def custom_feature_full(df):
    df = df.copy()

    monthly_income = df["AMT_INCOME_TOTAL"] / 12
    df["MONTHLY_INCOME_TO_PREV_CREDIT"] = monthly_income / (df["PREV_AMT_CREDIT_MEAN"] + 1)
    df["PREV_ANNUITY_TO_MONTHLY_INCOME"] = df["PREV_AMT_ANNUITY_MEAN"] / (monthly_income + 1)

    fix_member = df["CNT_FAM_MEMBERS"].fillna(1).replace(0, 1)
    df["INCOME_PER_FAMILY_MEMBER"] = df["AMT_INCOME_TOTAL"] / fix_member

    ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    df["EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)
    df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
    df["EXT_SOURCE_MAX"] = df[ext_cols].max(axis=1)
    df["EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)

    df["EXT_SOURCE_PROD"] = df["EXT_SOURCE_1"].fillna(df["EXT_SOURCE_1"].median())*df["EXT_SOURCE_2"].fillna(df["EXT_SOURCE_2"].median())*df["EXT_SOURCE_3"].fillna(df["EXT_SOURCE_3"].median())
    df["EXT_SOURCE_COUNT"] = df[ext_cols].notnull().sum(axis=1)

    return df
custom_feature_step = FunctionTransformer(custom_feature_full)

In [ ]:
cat_cols = set(X_train_full.select_dtypes(include = ['string', 'object', 'category']).columns)
cat_cols_list = list(cat_cols)

In [ ]:
te_cols = ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']
target_encoder_step = TargetEncoder(
    cols = te_cols,
    smoothing = 10.0,
    cv = skf.n_splits
)

In [ ]:
preprocessor = ColumnTransformer (
    transformers = [
        ("tar_encd_col", target_encoder_step, te_cols)
        ("cat_col", cat_with_na, cat_cols_list),
        ("custom_col", num_zero_transform, custom_fill_na_cols),
    ],
    remainder="passthrough",)

In [ ]:
prep_pipeline = Pipeline([
    ("custom_feature", custom_feature_step),
    ("de_fix", fix_de),
    ("prep", preprocessor)
])
X_train_prep = prep_pipeline.fit_transform(X_train_full)

In [ ]:
def objective_lgbm(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 20, 64),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 150),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 150, 500),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-2, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
    }

    clf_lgbm = lgb.LGBMClassifier(
        **params,
        random_state=17,
        n_jobs=4,
        verbose=-1
    )

    score = cross_val_score(clf_lgbm, X_train_prep, y_train, cv=skf, scoring='roc_auc', n_jobs=2).mean()
    return score

In [ ]:
study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=30)

[I 2026-09-30 20:51:39,866] A new study created in memory with name: no-name-63e1971f-d28f-4b31-9a0a-0d1941947f90
/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[I 2026-09-30 20:58:10,834] Trial 0 finished with value: 0.7781562157000085 and parameters: {'num_leaves': 61, 'min_child_samples': 127, 'learning_rate': 0.04785614027694871, 'n_estimators': 249, 'reg_alpha': 0.04579987339614466, 'reg_lambda': 0.14879004055639466}. Best is trial 0 with value: 0.7781562157000085.
[W 2026-09-30 21:01:42,183] Trial 1 failed with parameters: {'num_leaves': 31, 'min_child_samples': 21, 'learning_rate': 0.09040479257616761, 'n_estimators': 353, 'reg_alpha': 0.055007151107686676, 'reg_lambda': 0.23253757684310142} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  F

KeyboardInterrupt: 

In [ ]:
def objective_xgb(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 150, 450),
        "min_child_weight": trial.suggest_int("min_child_weight", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-2, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
    }

    clf_xgb = xgb.XGBClassifier(
        **params,
        tree_method="hist",
        device="cuda",
        random_state=17,
        n_jobs=4,
        verbosity=0,
        enable_categorical=True,
        eval_metric="auc"
    )

    score = cross_val_score(clf_xgb, X_train_prep, y_train, cv=skf, scoring='roc_auc', n_jobs=2).mean()
    return score

In [ ]:
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=30)

In [ ]:
best_params = {
    "lgbm": study_lgbm.best_params,
    "xgb": study_xgb.best_params
}

with open(f'{PROJECT_PATH}/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=4)

print("Найкращі параметри успішно збережено в best_params.json!")